In [ ]:
import json
import random
from pathlib import Path
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse, RedirectResponse
from fastapi.staticfiles import StaticFiles
from jinja2 import Environment, FileSystemLoader
from core.generators.player_generator import PlayerGenerator
from core.generators.team_generator import load_teams_from_json
from core.engine.match_simulator import simulate_match
from core.entities.player import Player, Position
from core.entities.team import Team

BASE_DIR = Path.cwd()
TEMPLATES_DIR = BASE_DIR / "interface" / "web" / "templates"
STATIC_DIR = BASE_DIR / "interface" / "web" / "static"

'''print(f"BASE_DIR: {BASE_DIR}")
print(f"TEMPLATES_DIR exists: {TEMPLATES_DIR.exists()}")
print(f"TEMPLATES_DIR: {TEMPLATES_DIR}")'''

app = FastAPI(title="Football Simulator")
app.mount("/static", StaticFiles(directory=str(STATIC_DIR)), name="static")
jinja_env = Environment(loader=FileSystemLoader(str(TEMPLATES_DIR)))

# Глобальные данные
TEAMS = load_teams_from_json()
players_db: list[Player] = []  # заменим на загрузку из файла
PLAYERS_FILE = BASE_DIR / "players.json"

def load_players() -> list[Player]:
    global players_db
    if not PLAYERS_FILE.exists():
        return []
    with open(PLAYERS_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    players_db = []
    for p_data in data:
        pos = next(pos for pos in Position if pos.value == p_data["position"])
        players_db.append(Player(
            id=p_data["id"],
            first_name=p_data["first_name"],
            last_name=p_data["last_name"],
            nation=p_data["nation"],
            age=p_data["age"],
            position=pos,
            rating=p_data["rating"],
            potential=p_data["potential"],
            team_id=p_data.get('team_id')
        ))
    return players_db

def save_players(players: list[Player]):
    data = []
    for p in players:
        data.append({
            "id": p.id,
            "first_name": p.first_name,
            "last_name": p.last_name,
            "nation": p.nation,
            "age": p.age,
            "position": p.position.value,
            "rating": p.rating,
            "potential": p.potential,
            'team_id': p.team_id
        })
    with open(PLAYERS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def render_template(template_name: str, context: dict) -> HTMLResponse:
    """Рендерит шаблон и возвращает HTMLResponse."""
    template = jinja_env.get_template(template_name)
    # Передаём request в контекст, если он есть
    return HTMLResponse(content=template.render(context))

load_players()

def update_teams_with_players():
    for team in TEAMS:
        team.players = [p for p in players_db if p.team_id == team.id]

update_teams_with_players()

@app.get("/", response_class=HTMLResponse)
async def index(request: Request):
    return render_template("index.html", {"request": request, 'teams': TEAMS})

@app.get("/teams", response_class=HTMLResponse)
async def list_teams(request: Request):
    return render_template("teams.html", {"request": request, 'teams': TEAMS})

@app.get("/team/{team_id}", response_class=HTMLResponse)
async def team_detail(request: Request, team_id: int):
    team = next((t for t in TEAMS if t.id == team_id), None)
    if  not team:
        return HTMLResponse('Team not found', status_code=404)
    team.players = [p for p in players_db if p.team_id == team_id]
    lineup = team.get_best_lineup()
    return render_template('team_detail.html', {
        'request': request,
        'team': team,
        'lineup': lineup
    })

@app.get("/players", response_class=HTMLResponse)
async def list_players(request: Request):
    return render_template("players.html", {
        "request": request,
        "players": players_db,
        "count": len(players_db),
        'teams': TEAMS
    })

@app.post("/generate", response_class=HTMLResponse)
async def generate_players(request: Request, count: int = Form(100)):
    gen = PlayerGenerator()
    next_id = max([p.id for p in players_db], default=0) + 1
    new_players = []
    for i in range(count):
        team = random.choice(TEAMS)
        player = gen.generate(next_id + i, team_id=team.id)
        new_players.append(player)
        team.players.append(player)

    players_db.extend(new_players)
    save_players(players_db)
    return render_template("players.html", {
        "request": request,
        "players": players_db,
        "count": len(players_db),
        'teams': TEAMS,
        "message": f"Сгенерировано {count} игроков"
    })

@app.get("/match", response_class=HTMLResponse)
async def match_form(request: Request):
    players = load_players()
    return render_template("match.html", {
        "request": request,
        "teams": TEAMS,
        "result": None
    })

@app.post("/simulate", response_class=HTMLResponse)
async def simulate(request: Request, home_team_id: int = Form(...), away_team_id: int = Form(...)):
    if  home_team_id == away_team_id:
        return render_template('match.html', {
            'request': request,
            'teams': TEAMS,
            'error': 'Нельзя выбрать одну и ту же команду'
        })
    home_team = next(t for t in TEAMS if t.id == home_team_id)
    away_team = next(t for t in TEAMS if t.id == away_team_id)

    home_team.players = [p for p in players_db if p.team_id == home_team_id]
    away_team.players = [p for p in players_db if p.team_id == away_team_id]

    if  len(home_team.players) < 11 or len(away_team.players) < 11:
        return render_template('march.html', {
            'request': Request,
            'teams': TEAMS,
            'errroe': 'В командах меньше 11 игроков'
        })
    

    home_goals, away_goals, home_lineup, away_lineup = simulate_match(home_team, away_team)
    result = {
        "home_goals": home_goals,
        "away_goals": away_goals,
        "home_team": home_team.name,
        "away_team": away_team.name,
        'home_lineup': [p.full_name() for p in home_lineup],
        'away_lineup': [p.full_name() for p in away_lineup]
    }
    return render_template("match.html", {
        "request": request,
        'teams': TEAMS,
        "result": result,
        'selected_home': home_team_id,
        'selected_away': away_team_id
    })